## Minimum Cost Bundle of McDonalds Menu Items

We first paste in our fdc api key and install our dependencies via requirements.txt:

In [1]:
apikey = "6k4anDq3cih8Tre4oqT28vAxd5dKKOAfKm53hdWx"

In [12]:
%pip install -r ../requirements.txt --upgrade -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
otter-grader 6.1.3 requires wrapt<2.0.0,>=1.16.0, but you have wrapt 2.1.1 which is incompatible.
ibis-framework 9.2.0 requires pandas<3,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


Import the necessary packages for pulling and processing data:

In [13]:
import pandas as pd
import fooddatacentral as fdc


## Defining and solving the Linear Program to minimize the cost of our Mcdonalds bundle

We try to minimize the price vector p:
$$
    \min_x p'x
$$

such that
$$\begin{bmatrix}
      A
   \end{bmatrix}x \geq \begin{bmatrix}
                        b_{min}
                      \end{bmatrix},$$

Where $A$ is the food to nutrient mapping matrix and $b$ is our vector of dietary reference intakes.

The function below optimizes our defined objective:


In [14]:
from  scipy.optimize import linprog as lp
import numpy as np
import warnings

import pandas as pd
import numpy as np
import warnings
from scipy.optimize import linprog as lp

import pandas as pd
import numpy as np
import warnings
from scipy.optimize import linprog as lp

def solve_subsistence_problem(FoodNutrients, Prices, dietmin, max_weight=None, tol=1e-6, drop_vitamins=None):
    """
    Solves Stigler's Subsistence Cost Problem using minimum requirements.
    """
    
    vitamin_map = {
        'A': 'Vitamin A, RAE',
        'E': 'Vitamin E (alpha-tocopherol)',
        'C': 'Vitamin C, total ascorbic acid'
    }

    if drop_vitamins is not None:
        if isinstance(drop_vitamins, str):
            drop_vitamins = [drop_vitamins]
            
        labels_to_drop = [vitamin_map[v.upper()] for v in drop_vitamins if v.upper() in vitamin_map]
        
        dietmin = dietmin.drop(labels=labels_to_drop, errors='ignore')

    
    try: 
        p = Prices.apply(lambda x: x.magnitude)
    except AttributeError: 
        p = Prices

    p = p.dropna()
    use = p.index.intersection(FoodNutrients.columns)
    p = p[use]

    A = FoodNutrients[p.index].fillna(0)
    A = A.loc[A.index.intersection(dietmin.index)]
    A = A.reindex(dietmin.index, axis=0)
    b = dietmin

    A = A.reindex(p.index, axis=1)
    A = A.reindex(b.index, axis=0)

    result = lp(p.values, -A.values, -b.values, method='highs')

    result.A = A
    result.b = b
    
    if result.success:
        result.diet = pd.Series(result.x, index=p.index)
    else:
        warnings.warn(result.message)
        result.diet = pd.Series(np.nan, index=p.index)

    return result
    
   

   

## Function to define b, the vector of recommended dietary intakes

In [15]:
def dietary_ref_intake(age, sex):
    """
    Given an age and sex, returns 2 pandas series corresponding to the recommended mininum dietary intakes and the recommended maximum dietary intakes.
    Upon calling the function, prints the 2 series to console. Users can index ([0] or [1]) to get either the minimum or the maximum dietary intakes respectively.
    """
    
    DRI_url = "https://docs.google.com/spreadsheets/d/1y95IsQ4HKspPW3HHDtH7QMtlDA66IUsCHJLutVL-MMc/"
    DRIs = read_sheets(DRI_url)

    # 1. Clean and Set Index
    diet_min = DRIs['diet_minimums'].set_index('Nutrition')
    diet_max = DRIs['diet_maximums'].set_index('Nutrition')


    gender_prefix = 'M' if sex.lower().startswith('m') else 'F'

    if age < 1: bucket = "0-1" 
    elif 1 <= age <= 3:  bucket = "1-3"
    elif 4 <= age <= 8:  bucket = "4-8"
    elif 9 <= age <= 13: bucket = "9-13"
    elif 14 <= age <= 18: bucket = "14-18"
    elif 19 <= age <= 30: bucket = "19-30"
    elif 31 <= age <= 50: bucket = "31-50"
    else: bucket = "51+"

    column_name = f"{gender_prefix} {bucket}"

    try:
        recom_min = diet_min[[column_name]]
        recom_max = diet_max[[column_name]]
    except KeyError:
        print(f"Column '{column_name}' not found in the spreadsheet.")
        return None, None

    return recom_min, recom_max

In [16]:
def solve_subsistence_for_demographic(FoodNutrients, Prices, age, sex, max_weight=None, tol=1e-6, drop_vitamins=None):
    """
    Wrapper function to solve the subsistence problem for a specific age and sex.
    Automatically fetches the Dietary Reference Intakes (min requirements) and formats them for the solver.
    """
    
    # 1. Fetch the dietary reference intakes for the given age and sex
    recom_min, recom_max = dietary_ref_intake(age, sex)
    
    # Handle the case where the demographic isn't found in the spreadsheet
    if recom_min is None:
        print("Could not solve: Dietary minimums not found for this age and sex.")
        return None
        
    # 2. Convert the returned single-column DataFrame into a pandas Series
    # .squeeze() safely converts a DataFrame with one column into a Series
    dietmin_series = recom_min.squeeze() 
    
    # 3. Pass the formatted data into the subsistence problem solver
    result = solve_subsistence_problem(
        FoodNutrients=FoodNutrients, 
        Prices=Prices, 
        dietmin=dietmin_series, 
        max_weight=max_weight, 
        tol=tol, 
        drop_vitamins=drop_vitamins
    )
    
    return result

## Defining the A matrix

We first load in our data as done below:

In [17]:
SHEETsUS = [
          ("https://docs.google.com/spreadsheets/d/10v5LV3p8Yto6Vh114b1tjRBt6tQzZOEORwSOP9WUl70/edit?gid=1449883614#gid=1449883614", "USA"),
         ]
SHEETsUK = [
          ("https://docs.google.com/spreadsheets/d/10v5LV3p8Yto6Vh114b1tjRBt6tQzZOEORwSOP9WUl70/edit?gid=1449883614#gid=1449883614", "UK"),
         ]
SHEETsCH = [
          ("https://docs.google.com/spreadsheets/d/10v5LV3p8Yto6Vh114b1tjRBt6tQzZOEORwSOP9WUl70/edit?gid=1449883614#gid=1449883614", "CHINA"),
         ]

In [18]:
from eep153_tools.sheets import read_sheets

dfUS = read_sheets(SHEETsUS[0][0])[SHEETsUS[0][1]]
dfUK = read_sheets(SHEETsUK[0][0])[SHEETsUK[0][1]]
dfCH = read_sheets(SHEETsCH[0][0])[SHEETsCH[0][1]]

Below is an example of the McDonalds Menu we are trying to optimize to get the cheapest bundle of meals:

In [19]:
dfUS

,Food,Price,Quantity,Units,FDC
0,Big Mac,5.99,2.2,hectogram,170720
1,Quarter Pounder with Cheese,5.79,2.0,hectogram,2706900
2,Filet-O-Fish,5.59,1.3,hectogram,170319
3,Hamburger,2.69,1.0,hectogram,2706924
4,Chicken McNuggets (4 pieces),3.19,0.6,hectogram,173297
5,French Fries (Small),2.89,0.7,hectogram,170721
6,Egg McMuffin,4.89,1.3,hectogram,173307
7,Low Fat Milk (1%),1.49,2.4,hectogram,2079727
8,Orange Juice,4.11,6.2,hectogram,2073232
9,Sausage McMuffin w/ Egg,5.99,1.7,hectogram,172068


In [20]:
D_US = {}
D_UK = {}
D_CH = {}

count = 0
for food in  dfUS.Food.tolist():
    try:
        FDC = dfUS.loc[dfUS.Food==food,:].FDC[count]
        count+=1
        D_US[food] = fdc.nutrients(apikey,FDC).Quantity
    except AttributeError: 
        warnings.warn("Couldn't find FDC Code %s for food %s." % (food,FDC))        

FoodNutrientsUS = pd.DataFrame(D_US,dtype=float)


count = 0
for food in  dfUK.Food.tolist():
    try:
        FDC = dfUK.loc[dfUK.Food==food,:].FDC[count]
        count+=1
        D_UK[food] = fdc.nutrients(apikey,FDC).Quantity
    except AttributeError: 
        warnings.warn("Couldn't find FDC Code %s for food %s." % (food,FDC))        

FoodNutrientsUK = pd.DataFrame(D_UK,dtype=float)

count = 0
for food in  dfCH.Food.tolist():
    try:
        FDC = dfCH.loc[dfCH.Food==food,:].FDC[count]
        count+=1
        D_CH[food] = fdc.nutrients(apikey,FDC).Quantity
    except AttributeError: 
        warnings.warn("Couldn't find FDC Code %s for food %s." % (food,FDC))        

FoodNutrientsCH = pd.DataFrame(D_CH,dtype=float)

Below is an example of our A matrix (specifically the food to nutrient mappings of the China menu):

In [21]:
FoodNutrientsCH.head()

,Big Mac,Filet-O-Fish,Double Cheeseburger,Cheeseburger,Chicken McNuggets (4 pieces),French Fries (Small),Low Fat Milk (1%),Orange Juice,Sausage McMuffin w/ Egg
"Alcohol, ethyl",NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN
Ash,1.87,1.95,NaN,NaN,2.19,1.91,NaN,NaN,2.34
Betaine,NaN,NaN,NaN,NaN,NaN,0.40,NaN,NaN,NaN
Caffeine,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN
"Calcium, Ca",116.00,120.00,179.0,123.0,11.00,19.00,104.0,146.0,172.00


In [22]:
dfUS['FDC Quantity'] = dfUS[['Quantity','Units']].T.apply(lambda x : fdc.units(x['Quantity'],x['Units']))
dfUS
# # Now may want to filter df by time or place--need to get a unique set of food names.
dfUS['FDC Price'] = (dfUS['Price'] / dfUS['Quantity'])

dfUS.dropna(how='any') # Drop food with any missing data

# # To use minimum price observed
PricesUS = dfUS.groupby('Food',sort=False)['FDC Price'].min()

In [23]:
dfUK['FDC Quantity'] = dfUK[['Quantity','Units']].T.apply(lambda x : fdc.units(x['Quantity'],x['Units']))
dfUK
# # Now may want to filter df by time or place--need to get a unique set of food names.
dfUK['FDC Price'] = (dfUK['Price'] / dfUK['Quantity'])

dfUK.dropna(how='any') # Drop food with any missing data

# # To use minimum price observed
PricesUK = dfUK.groupby('Food',sort=False)['FDC Price'].min()

In [24]:
dfCH['FDC Quantity'] = dfCH[['Quantity','Units']].T.apply(lambda x : fdc.units(x['Quantity'],x['Units']))
dfCH
# # Now may want to filter df by time or place--need to get a unique set of food names.
dfCH['FDC Price'] = (dfCH['Price'] / dfCH['Quantity'])

dfCH.dropna(how='any') # Drop food with any missing data

# # To use minimum price observed
PricesCH = dfCH.groupby('Food',sort=False)['FDC Price'].min()

Below is an example of p (in this example, the price vector of the Chinese Menu Items):

In [25]:
PricesCH

Food
Big Mac                         1.977273
Filet-O-Fish                    1.669231
Double Cheeseburger             2.400000
Cheeseburger                    1.890000
Chicken McNuggets (4 pieces)    3.150000
French Fries (Small)            1.457143
Low Fat Milk (1%)               0.620833
Orange Juice                    0.096774
Sausage McMuffin w/ Egg         0.511765
Name: FDC Price, dtype: float64

## Defining b

Here are the minimum dietary intake constraints of our diet:

In [26]:
recom_min, recom_max = dietary_ref_intake(21, 'M')
dietmin_series = recom_min.squeeze()
print(dietmin_series)

Nutrition
Energy                            2400.0
Protein                             56.0
Fiber, total dietary                33.6
Folate, DFE                        400.0
Calcium, Ca                       1000.0
Carbohydrate, by difference        130.0
Iron, Fe                             8.0
Magnesium, Mg                      400.0
Niacin                              16.0
Phosphorus, P                      700.0
Potassium, K                      4700.0
Riboflavin                           1.3
Thiamin                              1.2
Vitamin A, RAE                     900.0
Vitamin B-12                         2.4
Vitamin B-6                          1.3
Vitamin C, total ascorbic acid      90.0
Vitamin E (alpha-tocopherol)        15.0
Vitamin K (phylloquinone)          120.0
Zinc, Zn                            11.0
Name: M 19-30, dtype: float64


## Solving the subsistence Problem to get our minimum cost bundle of Mcdonalds menu items

### Minimum cost bundle for a 21 Year Old Male in the US

In [28]:
tol = 1e-6

result = solve_subsistence_for_demographic(FoodNutrientsUS,PricesUS,21,'M',tol=tol)

print("Cost of diet for 21 Year old US Male is $%4.2f per day.\n" % result.fun)

# Put back into nice series
diet = result.diet

print("\nDiet (in 100s of grams or milliliters):")
print(diet[diet >= tol])  # Drop items with quantities less than precision of calculation.
print()

tab = pd.DataFrame({"Outcome":np.abs(result.A).dot(diet),"Recommendation":np.abs(result.b)})
print("\nWith the following nutritional outcomes of interest:")
print(tab)
print()

print("\nConstraining nutrients are:")
excess = tab.diff(axis=1).iloc[:,1]
print(excess.loc[np.abs(excess) < tol*100].index.tolist())

Cost of diet for 21 Year old US Male is $86.25 per day.


Diet (in 100s of grams or milliliters):
Quarter Pounder with Cheese    17.647059
French Fries (Small)            8.312020
Orange Juice                    1.271952
dtype: float64


With the following nutritional outcomes of interest:
                                     Outcome  Recommendation
Nutrition                                                   
Energy                          16043.420290          2400.0
Protein                           297.635004            56.0
Fiber, total dietary               53.593350            33.6
Folate, DFE                       423.529412           400.0
Calcium, Ca                      3502.456948          1000.0
Carbohydrate, by difference       715.348133           130.0
Iron, Fe                           24.649616             8.0
Magnesium, Mg                     779.087809           400.0
Niacin                             98.658854            16.0
Phosphorus, P                    4285.

### Minimum cost bundle for a 21 Year Old Male in the US (Vitamin A,C,E constraints lifted)

In [29]:
result = solve_subsistence_for_demographic(FoodNutrientsUS,PricesUS,21,'M',tol=tol, drop_vitamins= ['A', 'C', 'E'])

print("Cost of diet for 21 Year old US Male is $%4.2f per day.\n" % result.fun)

# Put back into nice series
diet = result.diet

print("\nDiet (in 100s of grams or milliliters):")
print(diet[diet >= tol])  # Drop items with quantities less than precision of calculation.
print()

tab = pd.DataFrame({"Outcome":np.abs(result.A).dot(diet),"Recommendation":np.abs(result.b)})
print("\nWith the following nutritional outcomes of interest:")
print(tab)
print()

print("\nConstraining nutrients are:")
excess = tab.diff(axis=1).iloc[:,1]
print(excess.loc[np.abs(excess) < tol*100].index.tolist())


Cost of diet for 21 Year old US Male is $43.73 per day.


Diet (in 100s of grams or milliliters):
Hamburger               6.060606
French Fries (Small)    5.818182
Orange Juice            5.139394
dtype: float64


With the following nutritional outcomes of interest:
                                 Outcome  Recommendation
Nutrition                                               
Energy                       9696.533333          2400.0
Protein                       104.711758            56.0
Fiber, total dietary           33.600000            33.6
Folate, DFE                   400.000000           400.0
Calcium, Ca                  1563.927273          1000.0
Carbohydrate, by difference   482.791758           130.0
Iron, Fe                       22.048485             8.0
Magnesium, Mg                 400.000000           400.0
Niacin                         45.052024            16.0
Phosphorus, P                1405.575758           700.0
Potassium, K                 5627.781818         

### Minimum cost bundle for a 21 Year Old Male in the UK

In [31]:
tol = 1e-6

result = solve_subsistence_for_demographic(FoodNutrientsUK,PricesUK,21,'M',tol=tol)

print("Cost of diet for 21 Year old UK Male is $%4.2f per day.\n" % result.fun)

# Put back into nice series
diet = result.diet

print("\nDiet (in 100s of grams or milliliters):")
print(diet[diet >= tol])  # Drop items with quantities less than precision of calculation.
print()

tab = pd.DataFrame({"Outcome":np.abs(result.A).dot(diet),"Recommendation":np.abs(result.b)})
print("\nWith the following nutritional outcomes of interest:")
print(tab)
print()

print("\nConstraining nutrients are:")
excess = tab.diff(axis=1).iloc[:,1]
print(excess.loc[np.abs(excess) < tol*100].index.tolist())

Cost of diet for 21 Year old UK Male is $83.88 per day.


Diet (in 100s of grams or milliliters):
Quarter Pounder with Cheese    17.647059
French Fries (Small)            8.312020
Orange Juice                    1.271952
dtype: float64


With the following nutritional outcomes of interest:
                                     Outcome  Recommendation
Nutrition                                                   
Energy                          16043.420290          2400.0
Protein                           297.635004            56.0
Fiber, total dietary               53.593350            33.6
Folate, DFE                       423.529412           400.0
Calcium, Ca                      3502.456948          1000.0
Carbohydrate, by difference       715.348133           130.0
Iron, Fe                           24.649616             8.0
Magnesium, Mg                     779.087809           400.0
Niacin                             98.658854            16.0
Phosphorus, P                    4285.

### Minimum cost bundle for a 21 Year Old Male in the UK (Vitamin A,C,E Constraints lifted)

In [35]:
tol = 1e-6

result = solve_subsistence_for_demographic(FoodNutrientsUK,PricesUK,21,'M',tol=tol, drop_vitamins= ['A', 'C', 'E'])

print("Cost of diet for 21 Year old UK Male is $%4.2f per day.\n" % result.fun)

# Put back into nice series
diet = result.diet

print("\nDiet (in 100s of grams or milliliters):")
print(diet[diet >= tol])  # Drop items with quantities less than precision of calculation.
print()

tab = pd.DataFrame({"Outcome":np.abs(result.A).dot(diet),"Recommendation":np.abs(result.b)})
print("\nWith the following nutritional outcomes of interest:")
print(tab)
print()

print("\nConstraining nutrients are:")
excess = tab.diff(axis=1).iloc[:,1]
print(excess.loc[np.abs(excess) < tol*100].index.tolist())


Cost of diet for 21 Year old UK Male is $35.50 per day.


Diet (in 100s of grams or milliliters):
Hamburger               7.589055
French Fries (Small)    6.298400
dtype: float64


With the following nutritional outcomes of interest:
                                  Outcome  Recommendation
Nutrition                                                
Energy                       10511.357770          2400.0
Protein                        122.411977            56.0
Fiber, total dietary            38.224058            33.6
Folate, DFE                    500.877646           400.0
Calcium, Ca                   1000.000000          1000.0
Carbohydrate, by difference    492.821890           130.0
Iron, Fe                        26.819308             8.0
Magnesium, Mg                  400.000000           400.0
Niacin                          51.092411            16.0
Phosphorus, P                 1634.692824           700.0
Potassium, K                  5248.890036          4700.0
Riboflavin  

### Minimum cost bundle for a 21 Year Old Male in China

In [33]:
tol = 1e-6

result = solve_subsistence_for_demographic(FoodNutrientsCH,PricesCH,21,'M',tol=tol)

print("Cost of diet for 21 Year old Chinese Male is $%4.2f per day.\n" % result.fun)

# Put back into nice series
diet = result.diet

print("\nDiet (in 100s of grams or milliliters):")
print(diet[diet >= tol])  # Drop items with quantities less than precision of calculation.
print()

tab = pd.DataFrame({"Outcome":np.abs(result.A).dot(diet),"Recommendation":np.abs(result.b)})
print("\nWith the following nutritional outcomes of interest:")
print(tab)
print()

print("\nConstraining nutrients are:")
excess = tab.diff(axis=1).iloc[:,1]
print(excess.loc[np.abs(excess) < tol*100].index.tolist())


Cost of diet for 21 Year old Chinese Male is $45.09 per day.


Diet (in 100s of grams or milliliters):
Cheeseburger                16.363636
French Fries (Small)         1.875000
Orange Juice                 2.431818
Sausage McMuffin w/ Egg     21.879545
dtype: float64


With the following nutritional outcomes of interest:
                                     Outcome  Recommendation
Nutrition                                                   
Energy                          32139.004545          2400.0
Protein                           504.784727            56.0
Fiber, total dietary               58.095000            33.6
Folate, DFE                       638.181818           400.0
Calcium, Ca                      6166.679545          1000.0
Carbohydrate, by difference       901.525364           130.0
Iron, Fe                           81.420886             8.0
Magnesium, Mg                     863.888636           400.0
Niacin                            128.390727            16.0
Phos

### Minimum cost bundle for a 21 Year Old Male in China (Vitamin A,C,E Constraints lifted)

In [36]:
tol = 1e-6

result = solve_subsistence_for_demographic(FoodNutrientsCH,PricesCH,21,'M',tol=tol, drop_vitamins= ['A', 'C', 'E'])

print("Cost of diet for 21 Year old Chinese Male is $%4.2f per day.\n" % result.fun)

# Put back into nice series
diet = result.diet

print("\nDiet (in 100s of grams or milliliters):")
print(diet[diet >= tol])  # Drop items with quantities less than precision of calculation.
print()

tab = pd.DataFrame({"Outcome":np.abs(result.A).dot(diet),"Recommendation":np.abs(result.b)})
print("\nWith the following nutritional outcomes of interest:")
print(tab)
print()

print("\nConstraining nutrients are:")
excess = tab.diff(axis=1).iloc[:,1]
print(excess.loc[np.abs(excess) < tol*100].index.tolist())


Cost of diet for 21 Year old Chinese Male is $25.40 per day.


Diet (in 100s of grams or milliliters):
Cheeseburger            10.256410
French Fries (Small)     3.974359
Orange Juice             2.362248
dtype: float64


With the following nutritional outcomes of interest:
                                 Outcome  Recommendation
Nutrition                                               
Energy                       8251.227496          2400.0
Protein                       153.974768            56.0
Fiber, total dietary           34.987179            33.6
Folate, DFE                   400.000000           400.0
Calcium, Ca                  1681.939444          1000.0
Carbohydrate, by difference   456.349809           130.0
Iron, Fe                       26.256410             8.0
Magnesium, Mg                 406.571195           400.0
Niacin                         49.584064            16.0
Phosphorus, P                1961.153846           700.0
Potassium, K                 4700.000000 